# PE-AV Embedding Extraction

Extract audio, video, and joint (AV) embeddings from PE-AV model variants.
Configure `MODEL_IDS` below to run one or multiple variants.

**Environment:** `avtransformer`  
**Output:** `/outputs/model_embeddings/{model_name}/{bin_sec}s/{model_name}_{v,a,av}.npy`

In [ ]:
import gc
import os
import re
from pathlib import Path
import json
import numpy as np
import torch
from natsort import natsorted
from transformers import AutoModel, AutoProcessor

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from paths import DATA, OUTPUTS

# ── CONFIG ────────────────────────────────────────────────────────────────────
MODEL_CONFIGS = {
    "pe-av-small-16-frame": {
        "hf_id": "facebook/pe-av-small-16-frame",
        "modalities": ["v", "a", "av"],
    },
    # "pe-av-small":           {"hf_id": "facebook/pe-av-small",           "modalities": ["v", "a", "av"]},
    # "pe-av-base":            {"hf_id": "facebook/pe-av-base",            "modalities": ["v", "a", "av"]},
    # "pe-av-base-16-frame":   {"hf_id": "facebook/pe-av-base-16-frame",   "modalities": ["v", "a", "av"]},
    # "pe-av-large":           {"hf_id": "facebook/pe-av-large",           "modalities": ["v", "a", "av"]},
    # "pe-av-large-16-frame":  {"hf_id": "facebook/pe-av-large-16-frame",  "modalities": ["v", "a", "av"]},
}

SEGMENT_DURATIONS  = [5.0, 10.0]   # seconds
BATCH_SIZE         = 8
LOCAL_FILES_ONLY   = True
TEXT_MODALITIES    = ["avt", "at", "vt", "t"]

DATA_BASE   = DATA / "segmented_stimulus/filtered"
RUNS_BASE   = DATA / "runs"
OUTPUT_BASE = OUTPUTS / "model_embeddings"
TEXT_BASE   = OUTPUT_BASE / "text"
# ─────────────────────────────────────────────────────────────────────────────


## Helper Functions

In [ ]:
def get_model_shortname(model_id: str) -> str:
    """Convert HuggingFace model ID to a short filesystem-safe name."""
    return model_id.split("/")[-1]


def find_chunk_pairs(data_base: Path, seg_sec: float) -> list[tuple[Path, Path]]:
    '''Return sorted (video_mp4, audio_wav) pairs for a given segment duration.'''
    pairs = []
    for vid_subdir in natsorted(data_base.glob("Video*")):
        m = re.search(r"Video(\d+)$", vid_subdir.name)
        if not m:
            continue
        idx = m.group(1)
        chunk_dir = vid_subdir / f"Video{idx}_chunks_{int(seg_sec)}s"
        audio_dir = data_base / f"Audio{idx}" / f"Audio{idx}_chunks_{int(seg_sec)}s"
        if not chunk_dir.exists() or not audio_dir.exists():
            continue
        for vid in natsorted(chunk_dir.glob("*_part_*.mp4")):
            part_num = re.search(r"_part_(\d+)", vid.stem).group(1)
            aud = audio_dir / f"Audio{idx}_part_{part_num}.wav"
            if aud.exists():
                pairs.append((vid, aud))
    return pairs



@torch.no_grad()
def extract_embeddings(model, processor, pairs: list, batch_size: int,
                        device: str) -> dict[str, np.ndarray]:
    """Extract audio, video, and joint embeddings from video/audio file pairs.

    Returns dict with keys 'v', 'a', 'av' each containing (N, D) arrays.
    """
    video_embeds, audio_embeds, av_embeds = [], [], []

    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i + batch_size]
        vid_paths = [str(p[0]) for p in batch]
        aud_paths = [str(p[1]) for p in batch]

        inputs = processor(videos=vid_paths, audio=aud_paths, return_tensors="pt")
        inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                  for k, v in inputs.items()}

        outputs = model(**inputs)
        video_embeds.append(outputs.video_embeds.cpu().numpy())
        audio_embeds.append(outputs.audio_embeds.cpu().numpy())
        av_embeds.append(outputs.audio_video_embeds.cpu().numpy())

        if (i // batch_size) % 10 == 0:
            print(f"    {i + len(batch)}/{len(pairs)} chunks processed", end="\r")

    print()
    return {
        "v":  np.concatenate(video_embeds, axis=0),
        "a":  np.concatenate(audio_embeds, axis=0),
        "av": np.concatenate(av_embeds, axis=0),
    }

@torch.no_grad()
def extract_caption_t(model, processor, texts, batch_size, device):
    """Raw ModernBERT CLS for a list of text strings (text-only; no AV inputs).
    Adaptive batch size on CUDA OOM."""
    t_list = []
    cur_size = batch_size
    i = 0
    while i < len(texts):
        batch = texts[i : i + cur_size]
        try:
            enc = processor.tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128,
            ).to(device)
            out = model.text_model(
                input_ids=enc.input_ids,
                attention_mask=enc.attention_mask,
                output_hidden_states=True,
            )
            t_list.append(out.hidden_states[-1][:, 0].cpu().numpy())
            i += len(batch)
            print(f"    {i}/{len(texts)}", end="\r")
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            gc.collect()
            if cur_size == 1:
                raise
            cur_size = max(1, cur_size // 2)
            print(f"\n    OOM — reducing batch size to {cur_size}")
    print()
    return np.concatenate(t_list, axis=0)


@torch.no_grad()
def extract_avt_embeddings(model, processor, pairs, text_inputs, batch_size, device):
    """Extract text-conditioned AV embeddings; adaptive batch size on CUDA OOM.

    Returns a dict with keys 'avt', 'at', 'vt', 't', all shape (N, D):
      avt  — text_audio_video_embeds  (text CLS in AV contrastive space)
      at   — audio_plus_text_embeds
      vt   — video_plus_text_embeds
      t    — raw ModernBERT CLS (last hidden state [:, 0], before any head)
    """
    avt_list, at_list, vt_list, t_list = [], [], [], []
    cur_size = batch_size
    i = 0
    while i < len(pairs):
        batch      = pairs[i : i + cur_size]
        batch_text = text_inputs[i : i + cur_size]
        vid_paths  = [str(p[0]) for p in batch]
        aud_paths  = [str(p[1]) for p in batch]
        try:
            text_enc = processor.tokenizer(
                batch_text,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128,
            ).to(device)

            av_inputs = processor(videos=vid_paths, audio=aud_paths, return_tensors="pt")
            av_inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v
                         for k, v in av_inputs.items()}

            outputs = model(
                input_ids=text_enc.input_ids,
                attention_mask=text_enc.attention_mask,
                pixel_values_videos=av_inputs["pixel_values_videos"],
                input_values=av_inputs["input_values"],
                padding_mask_videos=av_inputs.get("padding_mask_videos"),
                padding_mask=av_inputs.get("padding_mask"),
            )

            avt_list.append(outputs.text_audio_video_embeds.cpu().numpy())
            at_list.append(outputs.audio_plus_text_embeds.cpu().numpy())
            vt_list.append(outputs.video_plus_text_embeds.cpu().numpy())
            t_list.append(outputs.text_outputs.hidden_states[-1][:, 0].cpu().numpy())

            i += len(batch)
            print(f"    {i}/{len(pairs)} processed (batch={cur_size})", end="\r")
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            gc.collect()
            if cur_size == 1:
                raise
            cur_size = max(1, cur_size // 2)
            print(f"\n    OOM — reducing batch size to {cur_size}")
    print()
    return {
        "avt": np.concatenate(avt_list, axis=0),
        "at":  np.concatenate(at_list,  axis=0),
        "vt":  np.concatenate(vt_list,  axis=0),
        "t":   np.concatenate(t_list,   axis=0),
    }


## Data Compatibility Check

Verify paired video/audio chunks exist for each segment duration.

In [ ]:
print("Chunk pair counts per segment duration:")
for seg_sec in SEGMENT_DURATIONS:
    pairs = find_chunk_pairs(DATA_BASE, seg_sec)
    print(f"  {seg_sec:4.1f}s → {len(pairs):4d} pairs")

## Embedding Extraction

Loop over all models in `MODEL_IDS`. Each model is loaded, all segment durations
are processed, and the model is deleted to free GPU memory before the next iteration.

In [ ]:
for model_name, cfg in MODEL_CONFIGS.items():
    model_id = cfg["hf_id"]
    modalities = cfg["modalities"]
    print(f"\n{'='*60}")
    print(f"Model: {model_id}")
    print(f"{'='*60}")

    # Load model
    processor = AutoProcessor.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY)
    model = AutoModel.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY).to(DEVICE)
    model.eval()

    for seg_sec in SEGMENT_DURATIONS:
        out_dir = OUTPUT_BASE / model_name / f"{int(seg_sec)}s"
        out_dir.mkdir(parents=True, exist_ok=True)

        # Skip if all modality files already exist
        if all((out_dir / f"{model_name}_{m}.npy").exists() for m in modalities):
            print(f"  {seg_sec}s — already done, skipping")
            continue

        print(f"  {seg_sec}s — collecting pairs ...")
        pairs = find_chunk_pairs(DATA_BASE, seg_sec)
        print(f"  {seg_sec}s — extracting from {len(pairs)} pairs ...")

        embeds = extract_embeddings(model, processor, pairs, BATCH_SIZE, DEVICE)

        for mod, arr in embeds.items():
            if mod not in modalities:
                continue
            out_path = out_dir / f"{model_name}_{mod}.npy"
            np.save(out_path, arr)
            print(f"    saved {out_path.name}  shape={arr.shape}")

    # Release GPU memory before loading next model
    del model, processor
    torch.cuda.empty_cache()
    gc.collect()
    print(f"  {model_name} complete, GPU memory released")

print("\nAll models done.")

## Full Movie Runs (Optional)

Extract embeddings from full-length movie runs (not pre-chunked segments).
Outputs are saved under `{model}/{bin_sec}s/full/`.

In [ ]:
EXTRACT_FULL_RUNS = False  # Set to True to run this cell

if EXTRACT_FULL_RUNS:
    for model_id in MODEL_IDS:
        short = get_model_shortname(model_id)
        processor = AutoProcessor.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY)
        model = AutoModel.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY).to(DEVICE)
        model.eval()

        for seg_sec in SEGMENT_DURATIONS:
            run_files = natsorted(RUNS_BASE.glob("*.mp4"))
            pairs = [(f, f.with_suffix(".wav")) for f in run_files
                     if f.with_suffix(".wav").exists()]

            out_dir = OUTPUT_BASE / short / f"{int(seg_sec)}s" / "full"
            out_dir.mkdir(parents=True, exist_ok=True)

            embeds = extract_embeddings(model, processor, pairs, BATCH_SIZE, DEVICE)
            for mod, arr in embeds.items():
                np.save(out_dir / f"{short}_{mod}.npy", arr)

        del model, processor
        torch.cuda.empty_cache()
        gc.collect()

## Caption Text Embeddings (`_caption_t`)

Extract raw ModernBERT CLS embeddings from the hand-labelled **"Clean concept"** sentences
in `Video_Audio_Pair_Concepts_All.xlsx` (one sentence per 5 s segment, 626 total).

Only the 5 s bin is processed — the Excel describes 5 s chunks and there is no per-segment
concept data for merged 10 s windows.

Output: `{OUTPUT_BASE}/{model_name}/bin5s_skip5s/{model_name}_caption_t.npy`  shape `(626, D)`

The embedding is identical in nature to `_t` from the AVT cells above: `text_outputs.hidden_states[-1][:, 0]`
(ModernBERT last-layer CLS, before any projection head), conditioned on the concept captions.

In [ ]:
import pandas as pd

CONCEPT_EXCEL = DATA / "Video_Audio_Pair_Concepts_All.xlsx"
CONCEPT_BIN   = "bin5s_skip5s"

df = pd.read_excel(CONCEPT_EXCEL)
df = df.sort_values(["Series", "Part"]).reset_index(drop=True)
concepts = df["Clean concept"].tolist()

expected_pairs = find_chunk_pairs(DATA_BASE, 5.0)
assert len(concepts) == len(expected_pairs), (
    f"Concept count ({len(concepts)}) != AV pair count ({len(expected_pairs)}). "
    "Check that the Excel covers all 5 s segments."
)
print(f"Concepts loaded: {len(concepts)} segments")
print(f"Example [0]: {concepts[0][:100]}")

FORCE_REEXTRACT_CAPTION = False

for model_name, cfg in MODEL_CONFIGS.items():
    model_id = cfg["hf_id"]
    print(f"\n{'='*60}\nModel: {model_id}\n{'='*60}")

    out_dir  = OUTPUT_BASE / model_name / CONCEPT_BIN
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{model_name}_caption_t.npy"

    if out_path.exists() and not FORCE_REEXTRACT_CAPTION:
        arr = np.load(out_path)
        print(f"  Already exists — skipping ({out_path.name}  shape={arr.shape})")
        continue

    torch.cuda.empty_cache()
    gc.collect()
    processor = AutoProcessor.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY)
    model     = AutoModel.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY).to(DEVICE)
    model.eval()

    arr = extract_caption_t(model, processor, concepts, BATCH_SIZE, DEVICE)
    np.save(out_path, arr)
    print(f"  Saved {out_path.name}  shape={arr.shape}")

    del model, processor
    torch.cuda.empty_cache()
    gc.collect()
    print(f"  {model_name} GPU memory released")

print("\nCaption text embeddings done.")


## Transcript Text Embeddings (`_transcript_t`)

Extract raw ModernBERT CLS embeddings from Whisper **transcripts only** (no visual input).
Uses the same text-only forward pass as `caption_t` but reads `transcripts.json`
instead of the concept Excel. Processed for all `SEGMENT_DURATIONS`.

Output: `{model_name}_transcript_t.npy` alongside the other modality files.

In [ ]:
FORCE_REEXTRACT_TRANSCRIPT = False

for model_name, cfg in MODEL_CONFIGS.items():
    model_id = cfg["hf_id"]
    print(f"\n{'='*60}\nModel: {model_id}\n{'='*60}")

    model     = None
    processor = None

    for seg_sec in SEGMENT_DURATIONS:
        out_dir  = OUTPUT_BASE / model_name / f"bin{int(seg_sec)}s_skip{int(seg_sec)}s"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{model_name}_transcript_t.npy"

        if out_path.exists() and not FORCE_REEXTRACT_TRANSCRIPT:
            arr = np.load(out_path)
            print(f"  bin{int(seg_sec)}s -- already exists, skipping ({out_path.name}  shape={arr.shape})")
            continue

        trn_file = TEXT_BASE / f"bin{int(seg_sec)}s_skip{int(seg_sec)}s" / "transcripts.json"
        if not trn_file.exists():
            print(f"  bin{int(seg_sec)}s -- transcripts.json not found, skipping")
            continue

        transcripts = json.loads(trn_file.read_text())
        expected_pairs = find_chunk_pairs(DATA_BASE, seg_sec)
        if len(transcripts) != len(expected_pairs):
            raise ValueError(
                f"bin{int(seg_sec)}s: {len(transcripts)} transcripts but "
                f"{len(expected_pairs)} AV pairs — re-run text.ipynb"
            )

        if model is None:
            torch.cuda.empty_cache()
            gc.collect()
            processor = AutoProcessor.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY)
            model     = AutoModel.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY).to(DEVICE)
            model.eval()

        print(f"  bin{int(seg_sec)}s -- extracting transcript_t from {len(transcripts)} segments ...")
        arr = extract_caption_t(model, processor, transcripts, BATCH_SIZE, DEVICE)
        np.save(out_path, arr)
        print(f"  Saved {out_path.name}  shape={arr.shape}")

    if model is not None:
        del model, processor
        torch.cuda.empty_cache()
        gc.collect()
        print(f"  {model_name} GPU memory released")

print("\nTranscript text embeddings done.")


## Event Text Embeddings (`_event_t`)

Extract raw ModernBERT CLS embeddings from **event descriptions**: concept captions
fused with Whisper transcripts via `rewrite_text_inputs.py`.

Reads `text/bin5s_skip5s/event_descriptions.json` (5 s only — concept captions
are defined at 5 s resolution). Run `rewrite_text_inputs.py` first to generate it.

Output: `{OUTPUT_BASE}/{model_name}/bin5s_skip5s/{model_name}_event_t.npy`  shape `(626, D)`

In [ ]:
FORCE_REEXTRACT_EVENT = False
EVENT_BIN = "bin5s_skip5s"

event_file = TEXT_BASE / EVENT_BIN / "event_descriptions.json"
if not event_file.exists():
    raise FileNotFoundError(
        f"{event_file}\n"
        "Run rewrite_text_inputs.py first to generate event_descriptions.json."
    )

event_texts    = json.loads(event_file.read_text())
expected_pairs = find_chunk_pairs(DATA_BASE, 5.0)
assert len(event_texts) == len(expected_pairs), (
    f"event_descriptions count ({len(event_texts)}) != AV pair count ({len(expected_pairs)}). "
    "Re-run rewrite_text_inputs.py."
)
print(f"Event descriptions loaded: {len(event_texts)} segments")
print(f"Example [0]: {event_texts[0][:100]}")

for model_name, cfg in MODEL_CONFIGS.items():
    model_id = cfg["hf_id"]
    print(f"\n{'='*60}\nModel: {model_id}\n{'='*60}")

    out_dir  = OUTPUT_BASE / model_name / EVENT_BIN
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{model_name}_event_t.npy"

    if out_path.exists() and not FORCE_REEXTRACT_EVENT:
        arr = np.load(out_path)
        print(f"  Already exists — skipping ({out_path.name}  shape={arr.shape})")
        continue

    torch.cuda.empty_cache()
    gc.collect()
    processor = AutoProcessor.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY)
    model     = AutoModel.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY).to(DEVICE)
    model.eval()

    arr = extract_caption_t(model, processor, event_texts, BATCH_SIZE, DEVICE)
    np.save(out_path, arr)
    print(f"  Saved {out_path.name}  shape={arr.shape}")

    del model, processor
    torch.cuda.empty_cache()
    gc.collect()
    print(f"  {model_name} GPU memory released")

print("\nEvent text embeddings done.")


## AVT Embeddings (`_transcript_avt`, `_event_avt`)

Feed `audio + video + text` through PE-AV using two different text sources.
Only `text_audio_video_embeds` is saved under each suffix.

| suffix | text source | bins |
|--------|-------------|------|
| `_transcript_avt` | `transcripts.json` (Whisper) | 5 s + 10 s |
| `_event_avt` | `event_descriptions.json` (concept + transcript) | 5 s only |

Run `rewrite_text_inputs.py` before extracting `_event_avt`.

In [ ]:
FORCE_REEXTRACT_TRANSCRIPT_AVT = False

for model_name, cfg in MODEL_CONFIGS.items():
    model_id = cfg["hf_id"]
    print(f"\n{'='*60}\nModel: {model_id}\n{'='*60}")

    model     = None
    processor = None

    for seg_sec in SEGMENT_DURATIONS:
        out_dir  = OUTPUT_BASE / model_name / f"bin{int(seg_sec)}s_skip{int(seg_sec)}s"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{model_name}_transcript_avt.npy"

        if out_path.exists() and not FORCE_REEXTRACT_TRANSCRIPT_AVT:
            arr = np.load(out_path)
            print(f"  bin{int(seg_sec)}s -- already exists, skipping ({out_path.name}  shape={arr.shape})")
            continue

        trn_file = TEXT_BASE / f"bin{int(seg_sec)}s_skip{int(seg_sec)}s" / "transcripts.json"
        if not trn_file.exists():
            print(f"  bin{int(seg_sec)}s -- transcripts.json not found, skipping")
            continue

        transcripts = json.loads(trn_file.read_text())
        pairs = find_chunk_pairs(DATA_BASE, seg_sec)
        if len(transcripts) != len(pairs):
            raise ValueError(
                f"bin{int(seg_sec)}s: {len(transcripts)} transcripts but "
                f"{len(pairs)} AV pairs — re-run text.ipynb"
            )

        if model is None:
            torch.cuda.empty_cache()
            gc.collect()
            processor = AutoProcessor.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY)
            model     = AutoModel.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY).to(DEVICE)
            model.eval()

        print(f"  bin{int(seg_sec)}s -- extracting transcript_avt from {len(pairs)} pairs ...")
        embeds = extract_avt_embeddings(model, processor, pairs, transcripts, BATCH_SIZE, DEVICE)
        np.save(out_path, embeds["avt"])
        print(f"  Saved {out_path.name}  shape={embeds['avt'].shape}")

    if model is not None:
        del model, processor
        torch.cuda.empty_cache()
        gc.collect()
        print(f"  {model_name} GPU memory released")

print("\nTranscript AVT embeddings done.")


In [ ]:
FORCE_REEXTRACT_EVENT_AVT = False
EVENT_AVT_BIN = "bin5s_skip5s"

event_file = TEXT_BASE / EVENT_AVT_BIN / "event_descriptions.json"
if not event_file.exists():
    raise FileNotFoundError(
        f"{event_file}\n"
        "Run rewrite_text_inputs.py first to generate event_descriptions.json."
    )

event_texts = json.loads(event_file.read_text())
pairs_5s    = find_chunk_pairs(DATA_BASE, 5.0)
assert len(event_texts) == len(pairs_5s), (
    f"event_descriptions count ({len(event_texts)}) != AV pair count ({len(pairs_5s)}). "
    "Re-run rewrite_text_inputs.py."
)
print(f"Event descriptions loaded: {len(event_texts)} segments")

for model_name, cfg in MODEL_CONFIGS.items():
    model_id = cfg["hf_id"]
    print(f"\n{'='*60}\nModel: {model_id}\n{'='*60}")

    out_dir  = OUTPUT_BASE / model_name / EVENT_AVT_BIN
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{model_name}_event_avt.npy"

    if out_path.exists() and not FORCE_REEXTRACT_EVENT_AVT:
        arr = np.load(out_path)
        print(f"  Already exists — skipping ({out_path.name}  shape={arr.shape})")
        continue

    torch.cuda.empty_cache()
    gc.collect()
    processor = AutoProcessor.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY)
    model     = AutoModel.from_pretrained(model_id, local_files_only=LOCAL_FILES_ONLY).to(DEVICE)
    model.eval()

    print(f"  Extracting event_avt from {len(pairs_5s)} pairs ...")
    embeds = extract_avt_embeddings(model, processor, pairs_5s, event_texts, BATCH_SIZE, DEVICE)
    np.save(out_path, embeds["avt"])
    print(f"  Saved {out_path.name}  shape={embeds['avt'].shape}")

    del model, processor
    torch.cuda.empty_cache()
    gc.collect()
    print(f"  {model_name} GPU memory released")

print("\nEvent AVT embeddings done.")
